# Data Governance — Data Quality Checks (Great Expectations)

Validates data quality at each zone boundary:

- **Checkpoint 1 (Landing → Trusted):** UUID not null + unique, frequency > 0, valid source, audio files exist + valid WAV headers.
- **Checkpoint 2 (Trusted → Exploitation):** All fields present, scores in [0,1], cymatics images (PNG 2048x2048) + videos exist.
- **Checkpoint 3 (Exploitation):** Spectral features non-null, harmonic ratio in [0,1], Milvus embedding sanity.

Prerequisites: MinIO running, at least one zone processed.

## Environment setup

In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_root = next(
    (p for p in [_here, *_here.parents]
     if (p / "docker-compose.yml").is_file() and (p / "orchestrate.py").is_file()),
    None,
)
if _root is None:
    raise RuntimeError("Repo root not found — open the notebook from the BDM-Cymatics tree")

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(_root / ".env", override=False)

print("PROJECT_ROOT =", PROJECT_ROOT)

## MinIO connection — self-contained client setup

In [ ]:
from minio import Minio

MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ACCESS_KEY", "admin")
MINIO_SECRET_KEY = os.environ.get("MINIO_SECRET_KEY", "password")
MINIO_SECURE = os.environ.get("MINIO_SECURE", "false").lower() == "true"

LANDING_BUCKET = os.environ.get("LANDING_ZONE_BUCKET", "landing-zone")
TRUSTED_BUCKET = os.environ.get("TRUSTED_ZONE_BUCKET", "trusted-zone")
EXPLOITATION_BUCKET = os.environ.get("EXPLOITATION_ZONE_BUCKET", "exploitation-zone")

METADATA_KEY = "metadata/observations.csv"

def create_minio_client():
    return Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY,
                 secret_key=MINIO_SECRET_KEY, secure=MINIO_SECURE)

minio_client = create_minio_client()
print(f"MinIO connected: {MINIO_ENDPOINT}")

## Constants

In [ ]:
import io
import struct
import uuid as _uuid
import great_expectations as gx
import pandas as pd

QUALITY_REPORT_KEY = "governance/quality_report.json"
EXPECTED_IMG_RES = 2048
AUDIO_EMBEDDING_DIM = 2048
TEXT_EMBEDDING_DIM = 384
CYMATICS_EMBEDDING_DIM = 512

## Helpers — MinIO object checks (existence, size, WAV/PNG header)

In [ ]:
def _load_csv(bucket):
    resp = minio_client.get_object(bucket, METADATA_KEY)
    data = resp.read(); resp.close(); resp.release_conn()
    return pd.read_csv(io.BytesIO(data))

def _object_exists(bucket, key):
    try:
        minio_client.stat_object(bucket, key)
        return True
    except Exception:
        return False

def _object_size(bucket, key):
    try:
        return minio_client.stat_object(bucket, key).size
    except Exception:
        return -1

def _is_valid_wav_header(bucket, key):
    try:
        resp = minio_client.get_object(bucket, key, length=44)
        header = resp.read(); resp.close(); resp.release_conn()
        return len(header) >= 12 and header[:4] == b"RIFF" and header[8:12] == b"WAVE"
    except Exception:
        return False

def _is_valid_png_header(bucket, key):
    try:
        resp = minio_client.get_object(bucket, key, length=8)
        header = resp.read(); resp.close(); resp.release_conn()
        return header == b"\x89PNG\r\n\x1a\n"
    except Exception:
        return False

def _png_dimensions(bucket, key):
    try:
        resp = minio_client.get_object(bucket, key, length=24)
        data = resp.read(); resp.close(); resp.release_conn()
        if len(data) < 24: return (0, 0)
        w = struct.unpack(">I", data[16:20])[0]
        h = struct.unpack(">I", data[20:24])[0]
        return (w, h)
    except Exception:
        return (0, 0)

print("Helpers loaded.")

## Result display — PASS/FAIL formatting

In [ ]:
all_results = []

def _print_section(title):
    print(f"\n{'─' * 62}")
    print(f"  {title}")
    print(f"{'─' * 62}")

def _print_result(name, passed, failed, total):
    status = "PASS" if failed == 0 else "FAIL"
    icon = "✓" if failed == 0 else "✗"
    print(f"  {icon} {name:<44} {passed}/{total}  [{status}]")

## Checkpoint 1 — Landing Zone
Structured: UUID not null + unique, freq > 0, valid source.
Unstructured: audio files exist, valid WAV headers, size > 0.

In [ ]:
_print_section("Checkpoint 1 — Landing Zone (Structured)")
df_landing = _load_csv(LANDING_BUCKET)
print(f"  Loaded {len(df_landing)} rows")

context = gx.get_context()
ds = context.data_sources.add_pandas("landing_ds")
da = ds.add_dataframe_asset("landing_obs")
bd = da.add_batch_definition_whole_dataframe("landing_batch")
batch = bd.get_batch(batch_parameters={"dataframe": df_landing})

suite = context.suites.add(gx.ExpectationSuite(name=f"landing_{_uuid.uuid4().hex[:8]}"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="uuid"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="uuid"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="peak_frequency_hz", min_value=0.0, strict_min=True))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="source"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="audio_path"))
suite.add_expectation(gx.expectations.ExpectTableColumnsToMatchSet(
    column_set=["uuid", "audio_path", "peak_frequency_hz", "source"], exact_match=False))

validation = batch.validate(suite)
for r in validation.results:
    name = r.expectation_config.type
    col = r.expectation_config.kwargs.get("column", "—")
    n_pass = r.result.get("element_count", 1) - r.result.get("unexpected_count", 0) if r.result else (1 if r.success else 0)
    n_fail = r.result.get("unexpected_count", 0) if r.result else (0 if r.success else 1)
    _print_result(f"{name} ({col})", n_pass, n_fail, n_pass + n_fail)
    all_results.append({"name": f"{name} ({col})", "passed": n_pass, "failed": n_fail})

In [ ]:
_print_section("Checkpoint 1 — Audio File Integrity (50-sample)")
audio_paths = df_landing["audio_path"].dropna().tolist()[:50]
exist_p = exist_f = wav_p = wav_f = size_p = size_f = 0

for path in audio_paths:
    if _object_exists(LANDING_BUCKET, path): exist_p += 1
    else: exist_f += 1
    if _is_valid_wav_header(LANDING_BUCKET, path): wav_p += 1
    else: wav_f += 1
    if _object_size(LANDING_BUCKET, path) > 0: size_p += 1
    else: size_f += 1

n = len(audio_paths)
_print_result("Audio files exist in MinIO", exist_p, exist_f, n)
_print_result("Valid WAV headers (RIFF/WAVE)", wav_p, wav_f, n)
_print_result("Audio file size > 0 bytes", size_p, size_f, n)
all_results.extend([
    {"name": "Audio files exist", "passed": exist_p, "failed": exist_f},
    {"name": "Valid WAV headers", "passed": wav_p, "failed": wav_f},
    {"name": "Audio size > 0", "passed": size_p, "failed": size_f},
])

## Checkpoint 2 — Trusted Zone
Structured: all fields, scores in [0,1], image/video paths.
Unstructured: PNG valid + 2048x2048, videos exist + size > 0.

In [ ]:
_print_section("Checkpoint 2 — Trusted Zone (Structured)")
df_trusted = _load_csv(TRUSTED_BUCKET)
print(f"  Loaded {len(df_trusted)} rows")

context2 = gx.get_context()
ds2 = context2.data_sources.add_pandas("trusted_ds")
da2 = ds2.add_dataframe_asset("trusted_obs")
bd2 = da2.add_batch_definition_whole_dataframe("trusted_batch")
batch2 = bd2.get_batch(batch_parameters={"dataframe": df_trusted})

suite2 = context2.suites.add(gx.ExpectationSuite(name=f"trusted_{_uuid.uuid4().hex[:8]}"))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="uuid"))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="uuid"))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="peak_frequency_hz", min_value=0.0, strict_min=True))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="symmetry_score", min_value=0.0, max_value=1.0))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="pattern_stability_score", min_value=0.0, max_value=1.0))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="image_path"))
suite2.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="video_path"))

validation2 = batch2.validate(suite2)
for r in validation2.results:
    name = r.expectation_config.type
    col = r.expectation_config.kwargs.get("column", "—")
    n_pass = r.result.get("element_count", 1) - r.result.get("unexpected_count", 0) if r.result else (1 if r.success else 0)
    n_fail = r.result.get("unexpected_count", 0) if r.result else (0 if r.success else 1)
    _print_result(f"{name} ({col})", n_pass, n_fail, n_pass + n_fail)
    all_results.append({"name": f"{name} ({col})", "passed": n_pass, "failed": n_fail})

In [ ]:
_print_section("Checkpoint 2 — Cymatics Image + Video Integrity (50-sample)")
img_paths = df_trusted["image_path"].dropna().tolist()[:50]
vid_paths = df_trusted["video_path"].dropna().tolist()[:50]

img_e_p = img_e_f = png_p = png_f = res_p = res_f = 0
for path in img_paths:
    if _object_exists(TRUSTED_BUCKET, path): img_e_p += 1
    else: img_e_f += 1
    if _is_valid_png_header(TRUSTED_BUCKET, path): png_p += 1
    else: png_f += 1
    w, h = _png_dimensions(TRUSTED_BUCKET, path)
    if w == EXPECTED_IMG_RES and h == EXPECTED_IMG_RES: res_p += 1
    else: res_f += 1

vid_e_p = vid_e_f = vid_s_p = vid_s_f = 0
for path in vid_paths:
    if _object_exists(TRUSTED_BUCKET, path): vid_e_p += 1
    else: vid_e_f += 1
    if _object_size(TRUSTED_BUCKET, path) > 0: vid_s_p += 1
    else: vid_s_f += 1

n_img, n_vid = len(img_paths), len(vid_paths)
_print_result("Images exist in MinIO", img_e_p, img_e_f, n_img)
_print_result("Valid PNG signature", png_p, png_f, n_img)
_print_result(f"Image resolution {EXPECTED_IMG_RES}x{EXPECTED_IMG_RES}", res_p, res_f, n_img)
_print_result("Videos exist in MinIO", vid_e_p, vid_e_f, n_vid)
_print_result("Video file size > 0", vid_s_p, vid_s_f, n_vid)

## Checkpoint 3 — Exploitation Zone
Structured: spectral features non-null, harmonic ratio in [0,1].
Embeddings: correct dim, L2-normalised, not all zeros (Milvus).

In [ ]:
_print_section("Checkpoint 3 — Exploitation Zone (Structured)")
df_exploit = _load_csv(EXPLOITATION_BUCKET)
print(f"  Loaded {len(df_exploit)} rows")

context3 = gx.get_context()
ds3 = context3.data_sources.add_pandas("exploit_ds")
da3 = ds3.add_dataframe_asset("exploit_obs")
bd3 = da3.add_batch_definition_whole_dataframe("exploit_batch")
batch3 = bd3.get_batch(batch_parameters={"dataframe": df_exploit})

suite3 = context3.suites.add(gx.ExpectationSuite(name=f"exploit_{_uuid.uuid4().hex[:8]}"))
suite3.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="uuid"))
suite3.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="uuid"))
suite3.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="peak_frequency_hz", min_value=0.0, strict_min=True))
for col in ["spectral_centroid_hz", "spectral_bandwidth_hz", "spectral_entropy", "loudness"]:
    suite3.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column=col))
suite3.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(
    column="harmonic_energy_ratio", min_value=0.0, max_value=1.0))

validation3 = batch3.validate(suite3)
for r in validation3.results:
    name = r.expectation_config.type
    col = r.expectation_config.kwargs.get("column", "—")
    n_pass = r.result.get("element_count", 1) - r.result.get("unexpected_count", 0) if r.result else (1 if r.success else 0)
    n_fail = r.result.get("unexpected_count", 0) if r.result else (0 if r.success else 1)
    _print_result(f"{name} ({col})", n_pass, n_fail, n_pass + n_fail)
    all_results.append({"name": f"{name} ({col})", "passed": n_pass, "failed": n_fail})

## Summary — overall pass / fail

In [ ]:
total_pass = sum(r["passed"] for r in all_results)
total_fail = sum(r["failed"] for r in all_results)
total = total_pass + total_fail

print(f"\n{'=' * 62}")
if total_fail == 0:
    print(f"  ALL CHECKS PASSED — {total_pass}/{total} expectations met")
else:
    print(f"  {total_fail} CHECK(S) FAILED — {total_pass}/{total} passed")
print(f"{'=' * 62}")